# Tarea 2: Question Answering Fine-tuning

In [1]:
# Librerías

import logging
logging.getLogger("transformers").setLevel(logging.ERROR)

import torch
print("Is CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Number of GPUs available:", torch.cuda.device_count())

from time import time
from datasets import *
from transformers import *
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
pd.set_option('display.colheader_justify', 'center')

Is CUDA available: True
CUDA version: 12.8
Number of GPUs available: 1


## Dataset

El dataset de SQuAD (Stanford Question Answering Dataset) es un conjunto de datos utilizado principalmente para entrenar y evaluar modelos de comprensión lectora. Consiste en ternas de preguntas, respuestas y contexto.

Aquí la ficha del dataset para que podáis explorarla: https://huggingface.co/datasets/rajpurkar/squad

In [2]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

dataset = load_dataset("squad")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

Con el único motivo de no demorar los tiempos de entrenamiento. Filtraremos el dataset y nos quedaremos solo con los registros que tenga longitud del campo _context_ inferior a 300.

El resto de la práctica se pide trabajarla sobre la variable `ds_tarea`.

In [3]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

def filtra_por_longitud(ejemplo):
    return len(ejemplo["context"]) < 300

ds_tarea = dataset.filter(filtra_por_longitud)

assert len(ds_tarea['train']) == 3466
assert len(ds_tarea['validation']) == 345

ds_tarea

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 3466
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 345
    })
})

## Modeling

En este apartado es donde tendréis que realizar todo el trabajo de la práctica. El formato, el análisis, el modelo escogido y cualquier proceso intermedio que consideréis es totalmente libre. Sin embargo, hay algunas pautas que tendréis que cumplir:

- La variable `model_checkpoint` debe almacenar el nombre del modelo y el tokenizador de 🤗 que vais a utilizar.
- La variable `model` y la variable `tokenizer` almacenarán, respectivamente, el modelo y el tokenizador de 🤗 que vais a utilizar.
- La variable `trainer` almacenará el _Trainer_ de 🤗 que, en la siguiente sección utilizaréis para entrenar el modelo.

In [4]:
# model_checkpoint = None
# tokenizer = None
# model = None
# trainer = None

### Análisis exploratorio del dataset filtrado

Antes de modelar, analizamos la distribución de longitudes para elegir correctamente los hiperparámetros de tokenización.

In [5]:
# Se analiza la longitud en caracteres de contextos, preguntas y respuestas
# Para ajustar el MAX_LENGTH y que el entrenamiento no gaste muchos recursos
ctx_lens  = [len(x) for x in ds_tarea['train']['context']]
q_lens    = [len(x) for x in ds_tarea['train']['question']]
ans_lens  = [len(x['text'][0]) for x in ds_tarea['train']['answers']]

stats = pd.DataFrame({
    'campo':       ['context (chars)', 'question (chars)', 'answer (chars)'],
    'min':         [min(ctx_lens),  min(q_lens),  min(ans_lens)],
    'mean':        [round(np.mean(ctx_lens),1), round(np.mean(q_lens),1), round(np.mean(ans_lens),1)],
    'p90':         [int(np.percentile(ctx_lens,90)), int(np.percentile(q_lens,90)), int(np.percentile(ans_lens,90))],
    'p95':         [int(np.percentile(ctx_lens,95)), int(np.percentile(q_lens,95)), int(np.percentile(ans_lens,95))],
    'max':         [max(ctx_lens),  max(q_lens),  max(ans_lens)],
})
print("Estadísticas de longitud")
print(stats.to_string(index=False))

print(f"\nEjemplo de muestra:")
ej = ds_tarea['train'][0]
print(f"context : {ej['context'][:120]}...")
print(f"question: {ej['question']}")
print(f"answer  : {ej['answers']['text'][0]}")

Estadísticas de longitud
     campo        min  mean  p90  p95  max
 context (chars) 151  234.7 290  295  299 
question (chars)  12   59.4  88   99  206 
  answer (chars)   1   15.6  31   41  112 

Ejemplo de muestra:
context : On February 6, 2016, one day before her performance at the Super Bowl, Beyoncé released a new single exclusively on musi...
question: Beyonce released the song "Formation" on which online music service?
answer  : Tidal


Viendo las estadísticas del dataset, se observa que para el percentil 95 el contexto mide menos de 295 caracteres y la pregunta menos de 100. Suponiendo que un token son unos 3 caracteres, nos quedaríamos en unos 133 tokens (400/3), por lo que un MAX_LENGTH = 140 es más que suficiente.

In [6]:
# En base al dataset y al max_length se selecciona 
model_checkpoint = "deepset/roberta-base-squad2"
# con su respectivo tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

loading configuration file config.json from cache at /home/usuario/.cache/huggingface/hub/models--deepset--roberta-base-squad2/snapshots/adc3b06f79f797d1c575d5479d6f5efe54a9e3b4/config.json
Model config RobertaConfig {
  "_name_or_path": "deepset/roberta-base-squad2",
  "architectures": [
    "RobertaForQuestionAnswering"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "language": "english",
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "name": "Roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.46.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading file vocab.json from cach

In [7]:
# Carga del modelo base
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

MAX_LENGTH = 140

def preprocess_examples(examples):
    # Se limpian espacios en blanco
    questions = [q.strip() for q in examples["question"]]
    # Se tokenizan pares de preguntas y contexto
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map     = inputs.pop("overflow_to_sample_mapping")
    answers        = examples["answers"]
    start_positions = []
    end_positions   = []

    for i, offset in enumerate(offset_mapping):
        sample_idx  = sample_map[i]
        answer      = answers[sample_idx]
        start_char  = answer["answer_start"][0]
        end_char    = start_char + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i) # Se identifican que tokens corresponden a la pregunta 
                                                # y cuales al contexto

        # Se localiza el inicio y fin del contexto en los tokens
        idx = 0
        while sequence_ids[idx] != 1:
            idx = 1 + idx
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx = 1 + idx
        context_end = idx - 1

        # Respuesta fuera del chunk -> posición (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    return inputs

# Se aplica la función de mapeo de manera que el modelo solo reciba tensores
train_dataset = ds_tarea["train"].map(
    preprocess_examples, batched=True,
    remove_columns=ds_tarea["train"].column_names,
)
validation_dataset = ds_tarea["validation"].map(
    preprocess_examples, batched=True,
    remove_columns=ds_tarea["validation"].column_names,
)

print(f"Train procesado     : {len(train_dataset)} ejemplos")
print(f"Validation procesado: {len(validation_dataset)} ejemplos")

loading configuration file config.json from cache at /home/usuario/.cache/huggingface/hub/models--deepset--roberta-base-squad2/snapshots/adc3b06f79f797d1c575d5479d6f5efe54a9e3b4/config.json
Model config RobertaConfig {
  "_name_or_path": "deepset/roberta-base-squad2",
  "architectures": [
    "RobertaForQuestionAnswering"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "language": "english",
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "name": "Roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.46.1",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}

loading weights file model.safete

Map:   0%|          | 0/3466 [00:00<?, ? examples/s]

Map:   0%|          | 0/345 [00:00<?, ? examples/s]

Train procesado     : 3466 ejemplos
Validation procesado: 345 ejemplos


In [8]:
# Configuración del entrenamiento y creación del Trainer
training_args = TrainingArguments(
    output_dir="./qa_roberta_squad2",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)

print("Trainer listo.")

PyTorch: setting up devices
The default value for the training argument `--report_to` will change in v5 (from all installed integrations to none). In v5, you will need to use `--report_to all` to get the same behavior as now. You should start updating your code and make this info disappear :-).
Using auto half precision backend


Trainer listo.


## Training

In [9]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

start = time()

trainer.train()

end = time()
print(f">>>>>>>>>>>>> elapsed time: {(end-start)/60:.0f}m")

***** Running training *****
  Num examples = 3,466
  Num Epochs = 3
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 651
  Number of trainable parameters = 124,056,578


  0%|          | 0/651 [00:00<?, ?it/s]

{'loss': 0.5368, 'grad_norm': 23.038440704345703, 'learning_rate': 1.8525345622119818e-05, 'epoch': 0.23}
{'loss': 0.5586, 'grad_norm': 18.7797794342041, 'learning_rate': 1.7019969278033796e-05, 'epoch': 0.46}
{'loss': 0.5576, 'grad_norm': 14.03990364074707, 'learning_rate': 1.5483870967741936e-05, 'epoch': 0.69}
{'loss': 0.5727, 'grad_norm': 23.75313949584961, 'learning_rate': 1.3947772657450078e-05, 'epoch': 0.92}



***** Running Evaluation *****
  Num examples = 345
  Batch size = 16


  0%|          | 0/22 [00:00<?, ?it/s]

...

  0%|          | 0/22 [00:00<?, ?it/s]

Saving model checkpoint to ./qa_roberta_squad2/checkpoint-651
Configuration saved in ./qa_roberta_squad2/checkpoint-651/config.json


{'eval_loss': 1.130771517753601, 'eval_runtime': 0.324, 'eval_samples_per_second': 1064.658, 'eval_steps_per_second': 67.891, 'epoch': 3.0}


Model weights saved in ./qa_roberta_squad2/checkpoint-651/model.safetensors
tokenizer config file saved in ./qa_roberta_squad2/checkpoint-651/tokenizer_config.json
Special tokens file saved in ./qa_roberta_squad2/checkpoint-651/special_tokens_map.json


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./qa_roberta_squad2/checkpoint-217 (score: 0.9197055101394653).


{'train_runtime': 56.3804, 'train_samples_per_second': 184.426, 'train_steps_per_second': 11.547, 'train_loss': 0.38969347906369034, 'epoch': 3.0}
>>>>>>>>>>>>> elapsed time: 1m


## Evaluation

In [10]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

print(f"**** EVALUACIÓN ****")
print(f"********\nTokenizer config:\n{tokenizer}")
print(f"\n\n********\nModel config:\n{model.config}")
print(f"\n\n********\nTrainer arguments:\n{trainer.args}")

**** EVALUACIÓN ****
********
Tokenizer config:
RobertaTokenizerFast(name_or_path='deepset/roberta-base-squad2', vocab_size=50265, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}


********
Model config:
RobertaConfig {

In [11]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

question_answerer = pipeline("question-answering", model=model, tokenizer=tokenizer, device=device)

In [12]:
# No modificar esta celda
# Esta celda, celda tiene que estar ejecutada en la entrega

assert len(ds_tarea['train']) == 3466
assert len(ds_tarea['validation']) == 345

def calculate_sentence_similarity(sentence1, sentence2):
    sentence1 = re.sub(r'[^a-zA-Z0-9\s]', '', sentence1).lower()
    sentence2 = re.sub(r'[^a-zA-Z0-9\s]', '', sentence2).lower()
    words1 = set(sentence1.lower().split())
    words2 = set(sentence2.lower().split())
    matches = len(words1.intersection(words2))
    total_words = len(words1.union(words2))
    if total_words == 0:
        return 0.0
    return (matches / total_words) * 100

samples = [324,342,249,176,70,168,120,58,90,192,278,289,197,146,323,248,260,273,112,211]
evaluation_list = []

for ii in samples:
    context = ds_tarea['validation'][ii]['context']
    question = ds_tarea['validation'][ii]['question']
    answer = ds_tarea['validation'][ii]['answers']
    answers = [f"{tt}" for ii, tt in enumerate(answer['text'])]
    prediction = question_answerer(context=context, question=question)['answer']
    match = max([calculate_sentence_similarity(w, prediction) for w in answers])
    evaluation_list.append((ii,context,question,answers,prediction,match))

print(f"*** evaluation_df ***")
evaluation_df = pd.DataFrame(evaluation_list, columns=['sample', 'context', 'question', 'real_answers', 'predicted_answer', 'match'])
evaluation_df[['sample','real_answers','predicted_answer', 'match']]

Disabling tokenizer parallelism, we're using DataLoader multithreading already
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


*** evaluation_df ***


,sample,real_answers,predicted_answer,match
0,324,"[Hospitality Business/Financial Centre, Downtown Riverside, Hospitality Business/Financial Centre]",Hospitality Business/Financial Centre,100.000000
1,342,"[Rugby, Rugby, Rugby]",Rugby,100.000000
2,249,"[extremely high, high, extremely high]",high,100.000000
3,176,"[""A Machine to End War"", ""A Machine to End War"", A Machine to End War]",A Machine to End War,100.000000
4,70,"[Death Wish Coffee, Death Wish Coffee, Death Wish Coffee]",Death Wish Coffee,100.000000
5,168,"[antagonistic, antagonistic, antagonistic]",antagonistic,100.000000
6,120,"[1892 to 1894, from 1892 to 1894, from 1892 to 1894]",1892 to 1894,100.000000
7,58,"[Vince Lombardi Trophy, the Vince Lombardi Trophy, Vince Lombardi Trophy]",Vince Lombardi Trophy,100.000000
8,90,"[5 Live Sports Extra, 5 Live Sports Extra, 5 Live Sports Extra]",5 Live Sports Extra,100.000000
9,192,"[time, time complexity, time complexity]",time complexity,100.000000


### Criterio de evaluación

La **nota final de la tarea2** estará relacionada con el resultado de las predicciones de vuestro modelo.

El criterio de evaluación será el siguiente:

- La tarea2 se aprobará si el notebook se entrega sin fallos y con un modelo entrenado (independientemente de sus predicciones).
- Se ponderará en función de la columna _match_, que otorga 100% de acierto si todas las palabras coinciden y bajará gradualmente el porcentaje de acierto en función del número de palabras que no coincidan.
    
Nota: La nota que se calcula a continuación es orientativa y podría verse reducida en función del código de la entrega.

In [13]:
print(f"Tu nota de la tarea2 es: {max(np.ceil(evaluation_df['match'].sum() / len(evaluation_df) / 10), 5.0)}")

Tu nota de la tarea2 es: 10.0
